# HW1-C. Building and Training the Model

## About this notebook

This notebook is part of HW1 for the 50.039 Deep Learning course at the Singapore University of Technology and Design.

**Author:** Matthieu DE MARI (matthieu_demari@sutd.edu.sg)

**Version:** 1.0 (2026)

**Requirements:**
- Python 3
- Matplotlib
- Numpy
- Pandas
- PyTorch
- Torchmetrics

## 0. Imports and CUDA

In [ ]:
# Matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
# Numpy
import numpy as np
# Pandas
import pandas as pd
# Torch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchmetrics.classification import BinaryAccuracy
# Helper functions (additional file)
from helper_functions import *

In [ ]:
# Use GPU if available, else use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Before We Start

Please copy-paste your completed `WaveDataset` class from Notebook A and your `GatedLayer` class from Notebook B into the cells below.

In [ ]:
# TODO: Paste your WaveDataset class from Notebook A here
class WaveDataset(Dataset):
    pass  # Replace with your implementation

In [ ]:
# TODO: Paste your GatedLayer class from Notebook B here
class GatedLayer(nn.Module):
    pass  # Replace with your implementation

Now let's reload the dataset and create the DataLoader.

In [ ]:
# Load and visualize the dataset
excel_file_path = 'wave_dataset.xlsx'
x1_list, x2_list, inputs, outputs = load_wave_dataset(excel_file_path=excel_file_path)

print("Input shape:", inputs.shape)
print("Output shape:", outputs.shape)
print("Class 0:", sum(outputs == 0), "samples")
print("Class 1:", sum(outputs == 1), "samples")

# Visualize
plot_wave_dataset(x1_list, x2_list, outputs, title="Training Dataset")

In [ ]:
# Create Dataset and DataLoader
# TODO: Update this with your solution from Notebook A
pt_dataset = WaveDataset()
batch_size = 64
pt_dataloader = DataLoader(pt_dataset, batch_size=batch_size, shuffle=True)

## 6. Building the Neural Network

Now we will build our complete neural network for the wave classification task.

### Architecture

The network will consist of:

1. **GatedLayer** (input: 2, output: 32) - Our custom gated layer from Notebook B
2. **Linear + ReLU** (input: 32, output: 64) - First hidden layer
3. **Linear + ReLU** (input: 64, output: 32) - Second hidden layer
4. **Linear + Sigmoid** (input: 32, output: 1) - Output layer for binary classification

```
Input (2) --> GatedLayer (32) --> Linear+ReLU (64) --> Linear+ReLU (32) --> Linear+Sigmoid (1)
```

In [ ]:
class WaveClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Layer 1: GatedLayer (2 -> 32)
        self.gated = None
        
        # Layer 2: Linear (32 -> 64)
        self.hidden1 = None
        
        # Layer 3: Linear (64 -> 32)
        self.hidden2 = None
        
        # Layer 4: Output layer (32 -> 1)
        self.output_layer = None
        
        # Activation functions
        self.relu = None
        self.sigmoid = None
        
        # Loss function and accuracy metric
        self.loss_fn = nn.BCELoss()
        self.accuracy_fn = BinaryAccuracy()
    
    def forward(self, x):
        # TODO: Implement the forward pass
        # 1. Pass through gated layer
        # 2. Pass through hidden1 + relu
        # 3. Pass through hidden2 + relu
        # 4. Pass through output_layer + sigmoid
        return None

---

**Question 11:** Show your completed code for the `WaveClassifier` class, including both `__init__` and `forward` methods.

---

In [ ]:
# Create the model and print its structure
model = WaveClassifier()
print(model)

In [ ]:
# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total_params}")

---

**Question 12:** About the output layer and activation:

- Why do we use **Sigmoid** (not Softmax) for the final activation in binary classification?
- What range of values does Sigmoid output?
- How do we interpret the output value as a class prediction? (i.e., when do we predict Class 0 vs Class 1?)

---

---

**Question 13:** About the loss function:

- What does **BCE** stand for in `nn.BCELoss()`?
- Write out the BCE loss formula for a single sample with true label $y$ (coming from dataset) and predicted probability $\hat{y}$.
- Why is BCE appropriate for binary classification? What would happen if we used Mean Squared Error (MSE) instead? Would the model still train?

---

## 7. Training the Model

Now we will train our model using gradient descent with the Adam optimizer.

Study the training loop below. Some values need to be filled in.

In [ ]:
# Move model to device (GPU if available)
model = WaveClassifier().to(device)

# Training hyperparameters
num_epochs = 100
# TODO: Set a reasonable learning rate
learning_rate = None

# Create optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training history for plotting
loss_history = []
accuracy_history = []

# Training loop
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    epoch_acc = 0.0
    num_batches = 0
    
    for batch in pt_dataloader:
        # Unpack batch
        inputs_batch, labels_batch = batch
        inputs_batch = inputs_batch.to(device)
        labels_batch = labels_batch.to(device).reshape(-1, 1)
        
        # TODO: Zero the gradients
        
        # TODO: Forward pass - get predictions
        
        # TODO: Compute loss
        
        # TODO: Backward pass
        
        # TODO: Update weights
        
        # Track metrics
        epoch_loss += loss.item()
        epoch_acc += model.accuracy_fn(predictions, labels_batch.int()).item()
        num_batches += 1
    
    # Average metrics for epoch
    avg_loss = epoch_loss / num_batches
    avg_acc = epoch_acc / num_batches
    loss_history.append(avg_loss)
    accuracy_history.append(avg_acc)
    
    # Print progress every 20 epochs
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {avg_acc:.4f}')

print("Training complete!")

---

**Question 14:** 

1. Complete the training loop above by uncommenting and filling in the necessary code.
2. Show your completed training loop code.
3. Report your final training accuracy (it should exceed **90%**).

---

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(loss_history)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True)

ax2.plot(accuracy_history)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training Accuracy')
ax2.grid(True)

plt.tight_layout()
plt.show()

---

**Question 15:** About `optimizer.zero_grad()`:

- What does `optimizer.zero_grad()` do?
- What would happen if we forgot to call it before each backward pass?
- At what point in the training loop should it be called?

---

### Visualizing the Learned Decision Boundary

In [ ]:
# Visualize the learned decision boundary
plot_decision_boundary(model, device=device)

## 8. Evaluating Generalization

Training accuracy tells us how well the model fits the training data, but it doesn't tell us if the model will work on **new, unseen data**.

To evaluate generalization, we need to test on data the model has never seen during training.

In [ ]:
# Load the unseen test dataset
test_file_path = 'unseen_wave_dataset.xlsx'
x1_test, x2_test, inputs_test, outputs_test = load_wave_dataset(excel_file_path=test_file_path)

print("Test dataset shape:", inputs_test.shape)
print("Class 0:", sum(outputs_test == 0), "samples")
print("Class 1:", sum(outputs_test == 1), "samples")

# Visualize test data
plot_wave_dataset(x1_test, x2_test, outputs_test, title="Test Dataset (Unseen)")

---

**Question 16:** Write code to evaluate your trained model on the unseen test dataset.

Your code should:
1. Convert the test data to PyTorch tensors
2. Set the model to evaluation mode (`model.eval()`)
3. Make predictions (use `torch.no_grad()` to disable gradient computation)
4. Calculate and report the **test accuracy**

Show your code and report the test accuracy.

---

In [ ]:
# TODO: Write your evaluation code here

# Convert test data to tensors
# test_inputs = ...
# test_labels = ...

# Set model to evaluation mode
# model.eval()

# Make predictions
# with torch.no_grad():
#     predictions = ...

# Calculate accuracy
# test_accuracy = ...
# print(f"Test Accuracy: {test_accuracy:.4f}")

---

**Question 17:** Compare your training accuracy and test accuracy:

- What is your training accuracy? What is your test accuracy?
- Is there a significant gap between them?
- Based on this gap, is your model:
  - **Overfitting** (high training acc, low test acc)?
  - **Underfitting** (low training acc, low test acc)?
  - **Generalizing well** (high training acc, high test acc)?
- How can you tell which case applies?

---

---

**Question 18:** If your model was overfitting, name **two techniques** from Week 1-4 that could improve generalization. For each technique:

1. Explain the principle behind how it works
2. How would you implement it in PyTorch?

---

## Bonus: Experimentation Space

Use the cells below to experiment with your model. Some ideas:
- Try different learning rates
- Change the network architecture
- Additional good practices (early stopping, saver/loader, regularization, feature engineering, etc.)

In [ ]:
# Your experimentation code here

## Submission Checklist

Before submitting, make sure you have answered all questions:

- [ ] Q1: ML problem description
- [ ] Q2: Decision boundary geometry
- [ ] Q3: WaveDataset code
- [ ] Q4: Dataset conceptual questions
- [ ] Q5: DataLoader code
- [ ] Q6: DataLoader conceptual questions
- [ ] Q7: Gate behavior analysis
- [ ] Q8: Differentiability and gradients
- [ ] Q9: GatedLayer code
- [ ] Q10: Comparison with Linear+ReLU
- [ ] Q11: WaveClassifier code
- [ ] Q12: Sigmoid activation
- [ ] Q13: BCE loss
- [ ] Q14: Training loop code and accuracy
- [ ] Q15: optimizer.zero_grad()
- [ ] Q16: Test evaluation code and accuracy
- [ ] Q17: Overfitting/underfitting analysis
- [ ] Q18: Regularization techniques